In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import datetime

In [ ]:
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from matplotlib import pyplot as plt

In [ ]:
import requests

In [ ]:
from scipy.stats import sigmaclip

---

In [ ]:
from shared_matplotlib_utils import get_figure
from shared_matplotlib_utils.border_funcs import fix_borders

----

In [ ]:
ENDPOINT = "http://homeassistant.local:8090"
ENDPOINT = "http://192.168.1.53:8090"

In [ ]:
frames = requests.get(ENDPOINT + "/frames/?filter=pull").json()
frames = frames.get("frames")
frames = {frame.get("id"): frame for frame in frames}

In [ ]:
assert frames

In [ ]:
frames_ids = list(frames.keys())
print(frame_ids[:10])

---

## Get voltages, format, and remove outliers

In [ ]:
def remove_outliers(values, low_sigma=3.0, high_sigma=3.0):
    """
    Removes outliers from a 1D array or list using sigma clipping.
    
    Parameters:
    - values: list or numpy array of numeric values
    - low_sigma, high_sigma: sigma thresholds for clipping
    
    Returns:
    - outlier_indices: list of indices in the original array that were outliers
    """
    
    values = np.asarray(values)
    # sigmaclip returns the clipped values, and lower/upper thresholds
    clean_vals, low, high = sigmaclip(values, low=low_sigma, high=high_sigma)
    
    # Find indices of outliers
    outlier_mask = ~np.isin(values, clean_vals)
    outlier_indices = np.where(outlier_mask)[0].tolist()
    
    return outlier_indices

In [ ]:
battery_status = {}

for frame_id in frame_ids:
    _result = requests.get(ENDPOINT + f"/frames/{frame_id}/battery?limit=5000").json()
    _readings = _result.get("readings")
    
    timestamps = [row.get("timestamp") for row in _readings]
    voltages = [row.get("voltage") for row in _readings]

    outliers = remove_outliers(voltages, low_sigma=5.0, high_sigma=5.0)

    if outliers:
        print("outliers:", [voltages[i] for i in outliers])
        timestamps = [value for i, value in enumerate(timestamps) if i not in outliers]
        voltages = [value for i, value in enumerate(voltages) if i not in outliers]

    timestamps = [int(datetime.datetime.strptime(date, "%Y-%m-%dT%H:%M:%S.%f").timestamp()) for date in timestamps]
    
    battery_status[frame_id] = dict()
    battery_status[frame_id]["timestamp"] = timestamps
    battery_status[frame_id]["voltage"] = voltages

    

In [ ]:
print(battery_status.get("e99c074f-da31-4582-8ef0-808adc02167d"))

In [ ]:
len(battery_status[frame_ids[0]].get("voltage"))

----

## Voltage over time

In [ ]:
fig, ax = get_figure()

day_zero = datetime.datetime.now().timestamp()

for frame_id in frame_ids:

    dates = battery_status.get(frame_id).get("timestamp")
    voltages = battery_status.get(frame_id).get("voltage")
    voltages = np.array(voltages)

    ax.plot(dates, voltages, ".")

def formatter_time_float(seconds, pos):
    """Automatically format time in seconds to nearest sec, mins, hours, days, years"""

    # Change to offset
    seconds = seconds - day_zero
    
    delta = datetime.timedelta(seconds=seconds)

    days = delta.days
    seconds = delta.seconds
    hours = seconds // 3600
    minutes = (seconds // 60) % 60

    if days > 0:
        return f"{days}d"

    if hours > 0:
        return f"{hours}h"

    if minutes > 0:
        return f"{minutes}m"

    return f"{seconds}s"

ax.xaxis.set_major_formatter(FuncFormatter(formatter_time_float))
fix_borders(ax)

----

## Per Day

In [ ]:
pdf = pd.concat(
    [pd.DataFrame(rows).assign(frame_id=group) for group, rows in battery_status.items()],
    ignore_index=True
)

In [ ]:
pdf

In [ ]:
pdf["day"] = pdf["timestamp"].map(lambda dt: datetime.datetime.fromtimestamp(dt).date())

In [ ]:
pdf

In [ ]:
groups_days = pdf.groupby(['frame_id', 'day'])['voltage'].apply(list)

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))

colors = plt.get_cmap('Set2').colors
colors = plt.get_cmap('tab10').colors

frame_ids = groups_days.index.get_level_values(0).unique()
days = sorted(groups_days.index.get_level_values(1).unique())

width = 0.8 / len(frame_ids)  # width of each box per group
x = np.arange(len(days))

zero_day = min(days)

for i, frame_id in enumerate(frame_ids):
    vals_per_day = []
    for day in days:

        # Safely get the voltage list for this frame_id/day
        try:
            vals = groups_days.loc[(frame_id, day)]
        except KeyError:
            vals = []
        vals_per_day.append(vals)
    
    positions = x + i*width  # shift boxes for this frame_id
    parts = ax.violinplot(vals_per_day, positions=positions, widths=width)

    # Color the violins
    color = colors[i]
    for pc in parts['bodies']:
        pc.set_facecolor(color)

# Set x-axis labels centered under grouped boxes

labels = [(day - zero_day).days for day in days]

ax.set_xticks(x + width*(len(frame_ids)-1)/2)
ax.set_xticklabels(labels)

# Legend
if False:
    handles = [Line2D([0], [0], color='w', markerfacecolor=colors[i], marker='s', markersize=10) for i in range(len(frame_ids))]
    ax.legend(handles, frame_ids, title="Frame ID", bbox_to_anchor=(1.05,1), loc='upper center')

fix_borders(ax)